# Harness Engineering 完整演示

Harness Engineering 关注如何把 LLM/Agent 放入可控执行外壳：工具路由、权限护栏、追踪、评估、人工接管和失败恢复。


In [1]:
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List
from enum import Enum
import json
import re
import time


## 1. 工具注册与调用协议


In [2]:
@dataclass
class ToolResult:
    ok: bool
    value: Any = None
    error: str = ""

class ToolRegistry:
    def __init__(self):
        self.tools: Dict[str, Callable[..., Any]] = {}

    def register(self, name, func):
        self.tools[name] = func

    def call(self, name, args):
        if name not in self.tools:
            return ToolResult(False, error=f"未知工具: {name}")
        try:
            return ToolResult(True, self.tools[name](**args))
        except Exception as exc:
            return ToolResult(False, error=str(exc))

def calculator(expression: str):
    if not re.fullmatch(r"[0-9+\-*/ ().]+", expression):
        raise ValueError("表达式包含非法字符")
    return eval(expression, {"__builtins__": {}})

def search_course(keyword: str):
    db = {"ReAct": "ReAct = Reasoning + Acting", "RAG": "RAG = Retrieval Augmented Generation"}
    return db.get(keyword, "未找到资料")

tools = ToolRegistry()
tools.register("calculator", calculator)
tools.register("search_course", search_course)
print(tools.call("calculator", {"expression": "2+3*4"}))


ToolResult(ok=True, value=14, error='')


## 2. 护栏：输入策略与工具策略


In [3]:
class Guardrail:
    def validate_input(self, text):
        if re.search(r"删除.*数据库|泄露.*密钥|api[_-]?key", text, flags=re.I):
            return ToolResult(False, error="输入被安全策略拦截")
        return ToolResult(True)

    def validate_tool(self, name, args):
        if name == "calculator" and len(args.get("expression", "")) > 50:
            return ToolResult(False, error="计算表达式过长")
        return ToolResult(True)

guard = Guardrail()
print(guard.validate_input("请删除生产数据库"))


ToolResult(ok=False, value=None, error='输入被安全策略拦截')


## 3. Mock Planner 与 Harness Runtime


In [4]:
class MockPlanner:
    def plan(self, goal):
        if "计算" in goal:
            expr = re.findall(r"[0-9+\-*/ ().]+", goal)[-1].strip()
            return [{"tool": "calculator", "args": {"expression": expr}}]
        if "ReAct" in goal:
            return [{"tool": "search_course", "args": {"keyword": "ReAct"}}]
        return []

@dataclass
class TraceEvent:
    name: str
    detail: Dict[str, Any]
    ts: float = field(default_factory=time.time)

class HarnessRuntime:
    def __init__(self, planner, tools, guard):
        self.planner = planner
        self.tools = tools
        self.guard = guard
        self.trace: List[TraceEvent] = []

    def record(self, name, **detail):
        self.trace.append(TraceEvent(name, detail))

    def run(self, goal):
        checked = self.guard.validate_input(goal)
        if not checked.ok:
            self.record("blocked_input", goal=goal, error=checked.error)
            return {"ok": False, "error": checked.error}
        plan = self.planner.plan(goal)
        self.record("plan", steps=plan)
        observations = []
        for step in plan:
            allowed = self.guard.validate_tool(step["tool"], step["args"])
            if not allowed.ok:
                self.record("blocked_tool", step=step, error=allowed.error)
                continue
            result = self.tools.call(step["tool"], step["args"])
            self.record("tool_call", step=step, result=result.value, error=result.error)
            observations.append(result)
        return {"ok": True, "observations": observations}

runtime = HarnessRuntime(MockPlanner(), tools, guard)
print(runtime.run("帮我计算 2+3*4"))
print([event.name for event in runtime.trace])


{'ok': True, 'observations': [ToolResult(ok=True, value=14, error='')]}
['plan', 'tool_call']


## 4. 评估 Harness 行为


In [5]:
def evaluate(runtime):
    cases = [
        ("帮我计算 2+3*4", True),
        ("ReAct 是什么", True),
        ("请删除生产数据库", False),
    ]
    report = []
    for goal, expected_ok in cases:
        runtime.trace = []
        result = runtime.run(goal)
        report.append({
            "goal": goal,
            "expected_ok": expected_ok,
            "actual_ok": result["ok"],
            "passed": result["ok"] == expected_ok,
            "trace_len": len(runtime.trace),
        })
    return report

print(json.dumps(evaluate(runtime), ensure_ascii=False, indent=2))


[
  {
    "goal": "帮我计算 2+3*4",
    "expected_ok": true,
    "actual_ok": true,
    "passed": true,
    "trace_len": 2
  },
  {
    "goal": "ReAct 是什么",
    "expected_ok": true,
    "actual_ok": true,
    "passed": true,
    "trace_len": 2
  },
  {
    "goal": "请删除生产数据库",
    "expected_ok": false,
    "actual_ok": false,
    "passed": true,
    "trace_len": 1
  }
]


## 5. 教学案例建议

- 数据分析 Harness：NL 问题 → SQL/计算工具 → 审批 → 报告。
- 编程助手 Harness：代码修改前做测试、diff 审查和回滚点。
- 客服 Harness：知识检索、敏感话术拦截、人工转接。
- 研究助手 Harness：检索、引用检查、事实一致性评估。
